# 📝 Cypher 심화 과제 LV1(기초): 경로·다중 조건

> 이 단원의 새 문법을 **하나씩** 확인합니다. 문제는 네 갈래로 묶여 있습니다.
>
> - **1. 경로 탐색**: 가변길이(`*1..2`·`*0..2`)·`shortestPath`·`length(p)`
> - **2. 조건으로 거르기**: `IN`·`CONTAINS`·`STARTS WITH`·`ENDS WITH`·`<>`·정규식 `=~`·`IN $names`·`OR` 묶기
> - **3. 관계로 거르기**: `WHERE` 에 패턴을 적어 관계가 **없는** 것과 **있는** 것을 거르기
> - **4. 정렬과 쪽 넘기기**: `ORDER BY`·`LIMIT`·`SKIP`

## 풀이 방법
1. 맨 위 **준비 셀 → 초기화 셀 → 시드 적재 셀**을 순서대로 실행하세요.
2. 각 문제의 **답안 셀**에 Cypher(문자열)를 채워 `run_cypher(...)` 로 실행하고, 결과를 지정된 변수에 담으세요. **자가채점 셀**로 확인합니다(✅ 통과!).

**도메인**: 대학 **선수과목** 그래프입니다. `(A)-[:PREREQ_OF]->(B)` 는 "**A 를 들어야 B 를 들을 수 있다**"(A 가 B 의 선수과목)를 뜻합니다.

화이팅!

아래 세 셀(연결 → 초기화 → 시드 적재)을 먼저 실행하세요.

In [ ]:
# [제공 코드] Neo4j 연결 (로컬 우선 -> 연결 불가 시 클라우드 Aura 자동 전환)
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase

# 1) 접속 정보 읽기 (.env)
load_dotenv(".env", override=True)
load_dotenv("../.env", override=True)
load_dotenv("내작업폴더/day28_Neo4j_설치_Movies/.env", override=True)

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "test0011")

AURA_URI = os.getenv("AURA_URI")
AURA_USER = os.getenv("AURA_USER")
AURA_PASSWORD = os.getenv("AURA_PASSWORD")

# 2) 스마트 드라이버 연결 (로컬 DB가 꺼져있으면 Aura Cloud DB로 자동 연결)
driver = None
try:
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    driver.verify_connectivity()
    print("✅ [연결 성공] 로컬 Neo4j Desktop:", NEO4J_URI)
except Exception:
    if AURA_URI and AURA_USER and AURA_PASSWORD:
        driver = GraphDatabase.driver(AURA_URI, auth=(AURA_USER, AURA_PASSWORD))
        driver.verify_connectivity()
        print("✅ [연결 성공] Neo4j Aura 클라우드 DB:", AURA_URI)
    else:
        raise ConnectionError("Neo4j에 연결할 수 없습니다. Neo4j Desktop을 켜거나 .env 설정을 확인하세요.")

# 3) 공용 헬퍼 함수
def run_cypher(query, **params):
    """Cypher 실행 -> 결과를 dict 리스트로 반환"""
    with driver.session() as session:
        return [record.data() for record in session.run(query, **params)]

> ⚠️ **아래 초기화 셀은 연결된 데이터베이스의 노드를 전부 지웁니다.** 앞 단원에서 만든 그래프(day28 의 Movies 예제, day29 의 과제 결과)도 함께 사라집니다. 되돌릴 수 없으니 `.env` 가 **실습 전용 DB** 를 가리키는지 먼저 확인하세요.

In [ ]:
# [제공 코드] 그래프 초기화: 실습 전용 DB 인지 꼭 확인하고 실행하세요! 노드·관계를 전부 지웁니다.
# MATCH (n) 은 모든 노드, DETACH 는 붙은 관계까지 함께 지우라는 뜻입니다.
run_cypher("MATCH (n) DETACH DELETE n")
print("초기화 완료. 남은 노드:", len(run_cypher("MATCH (n) RETURN n")))

오늘 풀 문제의 그래프입니다. 과목 9개가 선수과목 관계 9개로 이어져 있습니다.

<img src="images/선수과목_그래프.png" width="820">

In [ ]:
# [제공 코드] 대학 선수과목 시드 적재: 이 셀은 실행만 하세요(그래프를 처음부터 만듭니다).
run_cypher("""
CREATE (c5:Course {name:'CS302 운영체제', credits:4}),
       (c3:Course {name:'CS202 알고리즘', credits:4}),
       (c6:Course {name:'CS401 머신러닝', credits:4}),
       (m2:Course {name:'수학201 선형대수', credits:3}),
       (c1:Course {name:'CS101 프로그래밍입문', credits:3}),
       (m3:Course {name:'수학202 확률통계', credits:3}),
       (c2:Course {name:'CS201 자료구조', credits:3}),
       (m1:Course {name:'수학101 미적분학', credits:3}),
       (c4:Course {name:'CS301 데이터베이스', credits:3})
CREATE (c1)-[:PREREQ_OF]->(c2),
       (c2)-[:PREREQ_OF]->(c3),
       (c2)-[:PREREQ_OF]->(c4),
       (c2)-[:PREREQ_OF]->(c5),
       (c3)-[:PREREQ_OF]->(c6),
       (m1)-[:PREREQ_OF]->(m2),
       (m1)-[:PREREQ_OF]->(m3),
       (m2)-[:PREREQ_OF]->(c6),
       (m3)-[:PREREQ_OF]->(c6)
""")
print("선수과목 적재 완료. 과목:", len(run_cypher("MATCH (c:Course) RETURN c.name")), "개")


## 데이터 살펴보기
아래 셀은 **실행만** 하세요. 어떤 과목과 선수 관계가 있는지 훑어봅니다.

In [ ]:
# [제공 코드] 선수과목 관계를 먼저 훑어봅니다(실행만 하세요)
for r in run_cypher("MATCH (a:Course)-[:PREREQ_OF]->(b:Course) "
                    "RETURN a.name AS 선수과목, b.name AS 다음과목 ORDER BY a.name, b.name"):
    print(r['선수과목'], '→', r['다음과목'])

---
# 1. 경로 탐색

가변길이 패턴과 최단 경로를 하나씩 써 봅니다(교안_01).

## 1-1. 가변길이 경로: 이 과목 이후로 열리는 과목
**배경**: `CS101 프로그래밍입문` 을 들으면, 그 뒤로 1~2단계 안에 어떤 과목들을 들을 수 있게 될까요?

**요구사항**:
- `CS101 프로그래밍입문` 에서 `PREREQ_OF` 관계를 **앞 방향**으로 **1~2단계**(`*1..2`) 따라간 과목을 찾으세요.
- 결과를 변수 **`rows1_1`** 에 담으세요. 도착 과목 이름을 반환하되 중복은 `DISTINCT` 로 없애고, 반환 컬럼 별칭은 **`name`** 으로 하세요.

**예시**: 서로 다른 과목 **4개**가 나옵니다.

<details><summary>힌트</summary>

```text
접근방법:
- 시작 과목을 이름으로 특정하고, PREREQ_OF 를 가변길이로 앞으로 따라가 도착 과목을 모은다.

세부구현:
1. 시작 과목 노드를 이름으로 특정한다.
2. PREREQ_OF 관계를 앞 방향으로 1~2단계(가변길이 *1..2) 이어 도착 과목 노드를 잡는다.
3. 도착 과목 이름을 DISTINCT 로 중복 없이 반환한다(별칭 name). run_cypher 결과를 rows1_1 에 담는다.
```

</details>

In [ ]:
rows1_1 = run_cypher("""
MATCH (:Course {name: 'CS101 프로그래밍입문'})-[:PREREQ_OF*1..2]->(dest:Course)
RETURN DISTINCT dest.name AS name
""")

In [ ]:
# [자가채점]
assert sorted(r['name'] for r in rows1_1) == ['CS201 자료구조', 'CS202 알고리즘', 'CS301 데이터베이스', 'CS302 운영체제'], \
    '가변길이를 *1..2 로 줬는지(*1..1 이나 *1..3 이면 답이 달라집니다), 화살표 방향이 앞으로인지, 별칭이 name 인지 확인하세요'
print('✅ 통과!')

## 1-2. 가변길이 경로: 이 과목의 모든 선수과목
**배경**: 이번엔 반대로, `CS401 머신러닝` 을 들으려면 **먼저 들어야 하는 모든 과목**을 알고 싶습니다.

**요구사항**:
- 도착 과목을 `CS401 머신러닝` 으로 고정하고, 그리로 향하는 `PREREQ_OF` 관계를 **단계 제한 없이**(`*1..`) 거슬러, 그 앞에 오는 선행 과목들을 찾으세요.
- 결과를 변수 **`rows1_2`** 에 담으세요. 선행 과목 이름을 `DISTINCT` 로, 반환 컬럼 별칭은 **`name`** 으로.

**예시**: 서로 다른 선수과목 **6개**가 나옵니다(1-1 과 방향이 반대입니다).

<details><summary>힌트</summary>

```text
접근방법:
- 도착 과목을 이름으로 고정하고, 그리로 향하는 PREREQ_OF 사슬의 출발 과목들을 모은다.

세부구현:
1. 출발 과목은 열어 두고, 도착 과목만 CS401 머신러닝 으로 이름을 고정한다.
2. 둘을 PREREQ_OF 가변길이(*1.., 단계 제한 없음)로 잇는다. 1-1 과 패턴 방향은 같지만 고정하는 끝이 반대다.
3. 출발 과목 이름을 DISTINCT 로 반환한다(별칭 name). 결과를 rows1_2 에 담는다.
```

</details>

In [ ]:
rows1_2 = run_cypher("""
MATCH (pre:Course)-[:PREREQ_OF*1..]->(:Course {name: 'CS401 머신러닝'})
RETURN DISTINCT pre.name AS name
""")

In [ ]:
# [자가채점]
assert sorted(r['name'] for r in rows1_2) == ['CS101 프로그래밍입문', 'CS201 자료구조', 'CS202 알고리즘', '수학101 미적분학', '수학201 선형대수', '수학202 확률통계'], \
    '도착 과목을 CS401 머신러닝 으로 고정하고 출발 과목을 모았는지, 단계 제한을 열어 뒀는지(*1..) 확인하세요'
print('✅ 통과!')

## 1-3. 자기 자신까지 한 목록으로: 하한 `0`
**배경**: 수강 안내문에 "`CS101 프로그래밍입문` 과 그 뒤 두 단계 과목"을 **한 목록**으로 싣고 싶습니다. 1-1 의 답에는 `CS101` 자신이 빠져 있었습니다.

**요구사항**:
- 1-1 과 같은 쿼리에서 가변길이의 **하한만 `0` 으로** 바꿔(`*0..2`) 다시 찾으세요.
- 결과를 변수 **`rows1_3`** 에 담으세요. 과목 이름을 `DISTINCT` 로, 반환 컬럼 별칭은 **`name`** 으로.

**예시**: **5개**가 나옵니다. 1-1 의 4개에 **`CS101 프로그래밍입문` 자신**이 더해진 목록입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 1-1 의 쿼리를 그대로 두고 가변길이의 하한만 0 으로 내린다.

세부구현:
1. 1-1 과 같은 패턴을 적되 *1..2 를 *0..2 로 바꾼다.
2. 하한 0 은 '관계를 한 번도 안 타는 길이 0 인 경로' 를 허용해 출발 노드가 답에 들어온다.
3. 도착 과목 이름을 DISTINCT 로 반환한다(별칭 name). 결과를 rows1_3 에 담는다.
```

</details>

In [ ]:
rows1_3 = run_cypher("""
MATCH (:Course {name: 'CS101 프로그래밍입문'})-[:PREREQ_OF*0..2]->(dest:Course)
RETURN DISTINCT dest.name AS name
""")

In [ ]:
# [자가채점]
assert sorted(r['name'] for r in rows1_3) == ['CS101 프로그래밍입문', 'CS201 자료구조', 'CS202 알고리즘', 'CS301 데이터베이스', 'CS302 운영체제'], \
    '하한을 0 으로 내렸는지(*0..2), 별칭이 name 인지 확인하세요. CS101 자신이 목록에 들어와야 합니다'
print('✅ 통과!')

## 1-4. 최단 경로의 과목 순서
**배경**: `CS101 프로그래밍입문` 에서 `CS401 머신러닝` 까지 **가장 짧은 선수과목 사슬**은 어떤 과목들을 거칠까요?

**요구사항**:
- `shortestPath` 로 `CS101 프로그래밍입문` → `CS401 머신러닝` 경로를 구하고, 지나는 과목 이름을 **순서대로** 얻으세요.
- 결과를 변수 **`rows1_4`** 에 담으세요. 경로가 지나는 노드 이름 목록을 반환 컬럼 별칭 **`names`** 로 받으면, 결과의 첫 행 `rows1_4[0]['names']` 가 과목 목록입니다.

**예시**: 과목 **4개**를 순서대로 지나는 경로가 나옵니다.

<details><summary>힌트</summary>

```text
접근방법:
- shortestPath 로 경로 변수 p 를 잡고, nodes(p) 에서 이름만 뽑는다.

세부구현:
1. 두 과목을 이름으로 특정하고, PREREQ_OF 가변길이(*, 길이 제한 없음) 앞 방향 경로를 shortestPath 로 감싸 p 에 담는다.
2. nodes(p) 로 지나는 노드를 얻고, 리스트 표현으로 이름만 뽑아 별칭 names 로 반환한다.
3. 결과를 rows1_4 에 담는다.
```

</details>

In [ ]:
rows1_4 = run_cypher("""
MATCH p = shortestPath((start:Course {name: 'CS101 프로그래밍입문'})-[:PREREQ_OF*]->(end:Course {name: 'CS401 머신러닝'}))
RETURN [n IN nodes(p) | n.name] AS names
""")

In [ ]:
# [자가채점]
assert rows1_4[0]['names'] == ['CS101 프로그래밍입문', 'CS201 자료구조', 'CS202 알고리즘', 'CS401 머신러닝'], \
    'shortestPath 로 감쌌는지, nodes(p) 에서 이름만 뽑아 별칭 names 로 반환했는지 확인하세요'
print('✅ 통과!')

## 1-5. 최단 경로의 단계 수
**배경**: 이번엔 `수학101 미적분학` 에서 `CS401 머신러닝` 까지 **몇 단계**를 거쳐야 하는지 궁금합니다.

**요구사항**:
- `shortestPath` 로 `수학101 미적분학` → `CS401 머신러닝` 경로를 구하고, **`length(p)`**(관계 개수)만 구하세요.
- 결과를 변수 **`rows1_5`** 에 담으세요. `length(p)` 를 반환 컬럼 별칭 **`L`** 로 받으면 `rows1_5[0]['L']` 이 단계 수입니다.

**예시**: 단계 수(`length`)는 **2** 입니다(과목 3개를 지나면 관계는 2개).

<details><summary>힌트</summary>

```text
접근방법:
- shortestPath 로 경로 p 를 잡고 length(p) 만 반환한다.

세부구현:
1. 두 과목을 이름으로 특정하고 shortestPath 로 경로 p 를 잡는다(1-4 와 같은 방식).
2. 이번엔 노드 목록 대신 length(p) 만 별칭 L 로 반환한다.
3. 결과를 rows1_5 에 담는다.
```

</details>

In [ ]:
rows1_5 = run_cypher("""
MATCH p = shortestPath((start:Course {name: '수학101 미적분학'})-[:PREREQ_OF*]->(end:Course {name: 'CS401 머신러닝'}))
RETURN length(p) AS L
""")

In [ ]:
# [자가채점]
assert rows1_5[0]['L'] == 2, \
    'length(p) 를 별칭 L 로 반환했는지, 화살표 방향(->)을 지켰는지 확인하세요'
print('✅ 통과!')

## 1-6. 몇 단계 걸리나: `length(p)` 로 줄 세우기
**배경**: `CS101 프로그래밍입문` 을 들은 학생에게 "여기서 **가장 멀리 있는 과목**부터" 보여 주려 합니다. 과목마다 최단 몇 단계인지 재서 줄을 세웁니다.

**요구사항**:
- `CS101 프로그래밍입문` 에서 **도착 과목을 정하지 않고** `shortestPath` 를 구하세요(도착 자리를 `(b:Course)` 로 열어 둡니다).
- **출발 과목 자신은 빼세요**(`WHERE b.name <> 'CS101 프로그래밍입문'`). 넣으면 오류가 납니다.
- 결과를 변수 **`rows1_6`** 에 담으세요. 과목 이름을 **`name`**, `length(p)` 를 **`L`** 별칭으로 받고, **`L` 내림차순, 같으면 이름 오름차순**으로 정렬하세요.

**예시**: **5행**이 나오고 맨 위는 **CS401 머신러닝(3단계)** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 도착 노드에 이름을 박지 않으면 닿을 수 있는 모든 과목까지의 최단 경로가 한 행씩 나온다.

세부구현:
1. 출발 과목만 이름으로 특정하고 도착은 (b:Course) 로 열어 shortestPath 로 감싼다.
2. WHERE 로 출발 과목 자신을 뺀다(양 끝이 같으면 shortestPath 가 오류를 낸다).
3. b.name 을 name, length(p) 를 L 로 반환하고 ORDER BY L DESC, name 으로 정렬한다.
4. 결과를 rows1_6 에 담는다.
```

</details>

In [ ]:
rows1_6 = run_cypher("""
MATCH p = shortestPath((start:Course {name: 'CS101 프로그래밍입문'})-[:PREREQ_OF*]->(b:Course))
WHERE b.name <> 'CS101 프로그래밍입문'
RETURN b.name AS name, length(p) AS L
ORDER BY L DESC, name ASC
""")

In [ ]:
# [자가채점]
assert [(r['name'], r['L']) for r in rows1_6] == [('CS401 머신러닝', 3), ('CS202 알고리즘', 2), ('CS301 데이터베이스', 2), ('CS302 운영체제', 2), ('CS201 자료구조', 1)], \
    '도착 과목을 열어 뒀는지, 출발 과목 자신을 뺐는지, 별칭이 name·L 인지, L 내림차순과 이름 오름차순을 둘 다 줬는지 확인하세요'
print('✅ 통과!')

---
# 2. 조건으로 거르기

`WHERE` 에 쓰는 연산자를 하나씩 확인합니다(교안_02 1절).

## 2-1. 목록 안의 과목만: `IN`
**배경**: 관심 있는 과목 이름 목록을 주고, 그중 **실제로 존재하는** 과목만 확인합니다.

**요구사항**:
- 이름이 `['CS201 자료구조', 'CS301 데이터베이스', 'CS999 없는과목']` **목록 중 하나**(`IN`)인 과목을 찾으세요.
- 결과를 변수 **`rows2_1`** 에 담으세요. 과목 이름을 반환 컬럼 별칭 **`name`** 으로.

**예시**: 목록 3개 중 실제 존재하는 **2개**만 나옵니다(없는 과목은 빠집니다).

<details><summary>힌트</summary>

```text
접근방법:
- WHERE 절에서 이름이 주어진 목록 안에 있는지 IN 으로 검사한다.

세부구현:
1. 과목을 잡고, WHERE 에서 c.name 이 세 이름의 목록에 IN 되는지 거른다.
2. 과목 이름을 별칭 name 으로 반환한다. 결과를 rows2_1 에 담는다.
```

</details>

In [ ]:
rows2_1 = run_cypher("""
MATCH (c:Course)
WHERE c.name IN ['CS201 자료구조', 'CS301 데이터베이스', 'CS999 없는과목']
RETURN c.name AS name
""")

In [ ]:
# [자가채점]
assert sorted(r['name'] for r in rows2_1) == ['CS201 자료구조', 'CS301 데이터베이스'], \
    'IN 에 목록 세 개를 그대로 줬는지, 별칭이 name 인지 확인하세요(없는 과목은 자연히 빠집니다)'
print('✅ 통과!')

## 2-2. 이름 가운데의 글자로 찾기: `CONTAINS`
**배경**: 과목 이름은 `CS201 자료구조`·`수학201 선형대수` 처럼 **계열 표시(`CS`·`수학`) 뒤에 번호**가 붙습니다. 그래서 찾으려는 번호 `201` 은 이름의 맨 앞도 맨 뒤도 아닌 **가운데**에 놓입니다. 계열과 상관없이 **학수번호가 201번인 과목**을 모아 봅니다.

**요구사항**:
- 이름에 **`'201'` 이 포함**(`CONTAINS`)된 과목을 찾으세요.
- 결과를 변수 **`rows2_2`** 에 담으세요. 과목 이름을 반환 컬럼 별칭 **`name`** 으로.

**예시**: 계열이 다른 **2개** 과목이 나옵니다.

<details><summary>힌트</summary>

```text
접근방법:
- WHERE 절에서 이름에 '201' 이 들어가는지 CONTAINS 로 검사한다.
- 찾는 글자가 이름 맨 앞이 아니라 가운데 있다는 점에 주의한다.

세부구현:
1. 과목을 잡고, WHERE 에서 이름이 '201' 을 CONTAINS 하는지 거른다.
2. 과목 이름을 별칭 name 으로 반환한다. 결과를 rows2_2 에 담는다.
```

</details>

In [ ]:
rows2_2 = run_cypher("""
MATCH (c:Course)
WHERE c.name CONTAINS '201'
RETURN c.name AS name
""")

In [ ]:
# [자가채점]
assert sorted(r['name'] for r in rows2_2) == ['CS201 자료구조', '수학201 선형대수'], \
    'CONTAINS 로 이름 가운데를 봤는지 확인하세요. STARTS WITH 로는 0건입니다'
print('✅ 통과!')

## 2-3. 이름이 접두어로 시작하는 과목: `STARTS WITH`
**배경**: 수학 계열 과목은 이름이 `수학` 으로 시작합니다.

**요구사항**:
- 이름이 **`'수학'` 으로 시작**(`STARTS WITH`)하는 과목을 찾으세요.
- 결과를 변수 **`rows2_3`** 에 담으세요. 과목 이름을 반환 컬럼 별칭 **`name`** 으로.

**예시**: **3개** 과목이 나옵니다.

<details><summary>힌트</summary>

```text
접근방법:
- WHERE 절에서 이름이 '수학' 으로 시작하는지 STARTS WITH 로 검사한다.

세부구현:
1. 과목을 잡고, WHERE 에서 이름이 '수학' 을 STARTS WITH 하는지 거른다(맨 앞만 본다).
2. 과목 이름을 별칭 name 으로 반환한다. 결과를 rows2_3 에 담는다.
```

</details>

In [ ]:
rows2_3 = run_cypher("""
MATCH (c:Course)
WHERE c.name STARTS WITH '수학'
RETURN c.name AS name
""")

In [ ]:
# [자가채점]
assert sorted(r['name'] for r in rows2_3) == ['수학101 미적분학', '수학201 선형대수', '수학202 확률통계'], \
    'STARTS WITH 로 이름의 맨 앞을 봤는지, 별칭이 name 인지 확인하세요'
print('✅ 통과!')

## 2-4. 이름이 접미어로 끝나는 과목: `ENDS WITH`
**배경**: 이번엔 이름의 **끝**을 봅니다. 이름이 `'학'` 으로 끝나는 과목만 찾아 보세요.

**요구사항**:
- 이름이 **`'학'` 으로 끝나는**(`ENDS WITH`) 과목을 찾으세요.
- 결과를 변수 **`rows2_4`** 에 담으세요. 과목 이름을 반환 컬럼 별칭 **`name`** 으로.

**예시**: **1개** 과목이 나옵니다. 참고로 같은 글자를 2-2 처럼 `CONTAINS` 로 찾으면 **3개**가 나옵니다. 어디를 보느냐가 답을 가릅니다.

<details><summary>힌트</summary>

```text
접근방법:
- WHERE 절에서 이름이 그 글자로 끝나는지 ENDS WITH 로 검사한다.

세부구현:
1. 과목을 잡고, WHERE 에서 이름이 '학' 으로 ENDS WITH 하는지 거른다(맨 뒤만 본다).
2. 과목 이름을 별칭 name 으로 반환한다. 결과를 rows2_4 에 담는다.
```

</details>

In [ ]:
rows2_4 = run_cypher("""
MATCH (c:Course)
WHERE c.name ENDS WITH '학'
RETURN c.name AS name
""")

In [ ]:
# [자가채점]
assert sorted(r['name'] for r in rows2_4) == ['수학101 미적분학'], \
    'ENDS WITH 로 이름의 끝을 봤는지, 별칭이 name 인지 확인하세요. CONTAINS 를 쓰면 더 많이 잡힙니다'
print('✅ 통과!')

## 2-5. 같지 않은 것 고르기: `<>`
**배경**: 4학점짜리 무거운 과목 말고, **그 외 과목**만 보고 싶습니다.

**요구사항**:
- `credits` 가 **4가 아닌** 과목을 `<>` 로 찾으세요.
- 결과를 변수 **`rows2_5`** 에 담으세요. 과목 이름을 반환 컬럼 별칭 **`name`** 으로, **이름 오름차순** 정렬로.

**예시**: 이름 오름차순으로 **6개** 과목이 나옵니다.

<details><summary>힌트</summary>

```text
접근방법:
- WHERE 에서 credits 가 4가 아니라는 조건을 걸고, 이름순으로 정렬한다.

세부구현:
1. 과목을 잡고, WHERE 에서 credits 가 4가 아닌 과목만 남긴다.
2. 과목 이름을 별칭 name 으로 반환하고 ORDER BY 로 이름 오름차순 정렬한다. 결과를 rows2_5 에 담는다.
```

</details>

In [ ]:
rows2_5 = run_cypher("""
MATCH (c:Course)
WHERE c.credits <> 4
RETURN c.name AS name
ORDER BY name ASC
""")

In [ ]:
# [자가채점]
assert [r['name'] for r in rows2_5] == ['CS101 프로그래밍입문', 'CS201 자료구조', 'CS301 데이터베이스', '수학101 미적분학', '수학201 선형대수', '수학202 확률통계'], \
    '4학점이 아닌 과목만 남겼는지, ORDER BY 로 이름 오름차순 정렬했는지 확인하세요'
print('✅ 통과!')

## 2-6. 여러 패턴을 한 줄로: 정규식 `=~`
**배경**: CS 계열 과목은 이름이 `CS101 프로그래밍입문` 처럼 **영문 학수번호**로 시작합니다(수학 계열은 `수학101 미적분학` 처럼 한글로 시작합니다). 그런데 문서마다 `cs101` 처럼 소문자로 적히기도 합니다. **대소문자를 가리지 않고** CS 계열 과목을 모두 찾아 봅니다.

**요구사항**:
- 정규식 `=~` 와 **대소문자 무시 표시 `(?i)`** 를 써서, 이름이 **`cs` 로 시작**하는 과목을 찾으세요(`'(?i)cs.*'`).
- 결과를 변수 **`rows2_6`** 에 담으세요. 과목 이름을 반환 컬럼 별칭 **`name`** 으로.

**예시**: **6개**가 나옵니다. `(?i)` 를 빼면 **0개**가 되는 것도 확인해 보세요(실제 이름은 대문자 `CS` 로 시작하니까요).

<details><summary>힌트</summary>

```text
접근방법:
- WHERE 절에서 이름이 정규식과 맞는지 =~ 로 검사한다.

세부구현:
1. 과목을 잡고 WHERE c.name =~ '(?i)cs.*' 를 적는다.
2. (?i) 는 맨 앞에 붙이는 대소문자 무시 표시이고, .* 는 '뒤는 무엇이든' 을 뜻한다.
3. 과목 이름을 별칭 name 으로 반환한다. 결과를 rows2_6 에 담는다.
```

</details>

In [ ]:
rows2_6 = run_cypher("""
MATCH (c:Course)
WHERE c.name =~ '(?i)cs.*'
RETURN c.name AS name
""")

In [ ]:
# [자가채점]
assert sorted(r['name'] for r in rows2_6) == ['CS101 프로그래밍입문', 'CS201 자료구조', 'CS202 알고리즘', 'CS301 데이터베이스', 'CS302 운영체제', 'CS401 머신러닝'], \
    "=~ 로 정규식을 줬는지, 맨 앞에 (?i) 를 붙였는지, 별칭이 name 인지 확인하세요"
print('✅ 통과!')

## 2-7. 목록을 파라미터로 넘겨 거르기: `IN $names`
**배경**: 2-1 에서는 이름 목록을 **쿼리 안에** 적었습니다. 실제 프로그램에서는 그 목록이 **파이썬 쪽에서** 정해집니다(사용자가 화면에서 고른 과목 같은 것). 쿼리는 그대로 두고 값만 넘깁니다.

**요구사항**:
- 파이썬 변수에 목록 `['CS201 자료구조', '수학201 선형대수', 'CS999 없는과목']` 를 담으세요.
- 쿼리에는 `WHERE c.name IN $names` 라고만 적고, `run_cypher(..., names=목록)` 으로 넘기세요.
- 결과를 변수 **`rows2_7`** 에 담으세요. 과목 이름을 반환 컬럼 별칭 **`name`** 으로.

**예시**: 목록 3개 중 실제 존재하는 **2개**만 나옵니다.

<details><summary>힌트</summary>

```text
접근방법:
- 목록을 파이썬 변수에 담고 $names 자리표시자로 넘긴다.

세부구현:
1. 목록을 파이썬 리스트 변수로 만든다.
2. 쿼리에는 WHERE c.name IN $names 라고 적는다(목록을 문자열로 이어 붙이지 않는다).
3. run_cypher(쿼리, names=목록) 으로 실행하고 결과를 rows2_7 에 담는다.
```

</details>

In [ ]:
target_names = ['CS201 자료구조', '수학201 선형대수', 'CS999 없는과목']
rows2_7 = run_cypher("""
MATCH (c:Course)
WHERE c.name IN $names
RETURN c.name AS name
""", names=target_names)

In [ ]:
# [자가채점]
assert sorted(r['name'] for r in rows2_7) == ['CS201 자료구조', '수학201 선형대수'], \
    '$names 자리표시자를 썼는지, run_cypher 에 names=목록 으로 넘겼는지, 별칭이 name 인지 확인하세요'
print('✅ 통과!')

## 2-8. 두 조건을 `OR` 로 묶기
**배경**: 수강 편람에 "**수학 계열이거나, 학점이 큰 과목**"을 한 목록으로 싣습니다. 둘 중 **하나만** 맞아도 실어야 합니다.

**요구사항**:
- 이름이 **`'수학'` 으로 시작**하거나(`STARTS WITH`) **`credits` 가 4**인 과목을 `OR` 로 묶어 찾으세요.
- 결과를 변수 **`rows2_8`** 에 담으세요. 과목 이름을 반환 컬럼 별칭 **`name`** 으로.

**예시**: **6개**가 나옵니다. 같은 두 조건을 `AND` 로 묶으면 **0개**가 됩니다(수학 계열이면서 4학점인 과목은 없으니까요).

<details><summary>힌트</summary>

```text
접근방법:
- 서로 다른 두 조건을 한 WHERE 에 OR 로 나란히 적는다.

세부구현:
1. 과목을 잡고 WHERE 에 이름 접두 조건과 학점 조건을 적는다.
2. 두 조건 사이를 AND 가 아니라 OR 로 잇는다(하나만 맞아도 남는다).
3. 과목 이름을 별칭 name 으로 반환한다. 결과를 rows2_8 에 담는다.
```

</details>

In [ ]:
rows2_8 = run_cypher("""
MATCH (c:Course)
WHERE c.name STARTS WITH '수학' OR c.credits = 4
RETURN c.name AS name
""")

In [ ]:
# [자가채점]
assert sorted(r['name'] for r in rows2_8) == ['CS202 알고리즘', 'CS302 운영체제', 'CS401 머신러닝', '수학101 미적분학', '수학201 선형대수', '수학202 확률통계'], \
    '두 조건을 AND 가 아니라 OR 로 묶었는지, 별칭이 name 인지 확인하세요 (AND 로 묶으면 0개가 나옵니다)'
print('✅ 통과!')

---
# 3. 관계로 거르기

값이 아니라 **관계가 있느냐 없느냐**로 거릅니다(교안_02 2절).

## 3-1. 선수과목이 하나도 없는 과목
**배경**: 신입생 안내문에 "**아무 것도 먼저 듣지 않고 바로 들을 수 있는 과목**"을 싣습니다. 즉 자기 앞에 `PREREQ_OF` 로 들어오는 화살표가 **하나도 없는** 과목입니다.

**요구사항**:
- 과목을 잡고, `WHERE` 에 **패턴을 그대로 적어** 선수과목이 없는 것만 남기세요(`WHERE NOT (:Course)-[:PREREQ_OF]->(c)`).
- 결과를 변수 **`rows3_1`** 에 담으세요. 과목 이름을 반환 컬럼 별칭 **`name`** 으로.

**예시**: **2개**가 나옵니다.

> `WHERE` 에 적는 패턴에는 **새 변수를 만들 수 없습니다.** 선수과목의 이름이 필요 없으니 `(:Course)` 처럼 레이블만 적으세요.

<details><summary>힌트</summary>

```text
접근방법:
- 관계가 '없는' 것은 MATCH 로 이어서는 찾을 수 없다. WHERE 에 패턴을 적고 NOT 으로 뒤집는다.

세부구현:
1. MATCH (c:Course) 로 과목을 모두 잡는다.
2. WHERE NOT (:Course)-[:PREREQ_OF]->(c) 를 적는다. 들어오는 화살표가 하나도 없는 과목만 남는다.
3. 과목 이름을 별칭 name 으로 반환한다. 결과를 rows3_1 에 담는다.
```

</details>

In [ ]:
rows3_1 = run_cypher("""
MATCH (c:Course)
WHERE NOT (:Course)-[:PREREQ_OF]->(c)
RETURN c.name AS name
""")

In [ ]:
# [자가채점]
assert sorted(r['name'] for r in rows3_1) == ['CS101 프로그래밍입문', '수학101 미적분학'], \
    'WHERE 에 패턴을 적고 NOT 으로 뒤집었는지, 화살표 방향이 (:Course)-[:PREREQ_OF]->(c) 인지, 별칭이 name 인지 확인하세요'
print('✅ 통과!')

## 3-2. 뒤에 이어지는 과목이 있는 과목
**배경**: 3-1 은 `NOT` 을 붙여 "관계가 **없는**" 것을 골랐습니다. 이번에는 `NOT` 없이 "관계가 **있는**" 것을 고릅니다. 이 과목을 들으면 **그다음에 열리는 과목이 하나라도 있는** 과목을 찾습니다.

**요구사항**:
- 과목을 잡고, `WHERE` 에 **패턴을 그대로 적어**(`WHERE (c)-[:PREREQ_OF]->(:Course)`) 다음 과목이 있는 것만 남기세요. `NOT` 은 붙이지 않습니다.
- 결과를 변수 **`rows3_2`** 에 담으세요. 과목 이름을 반환 컬럼 별칭 **`name`** 으로.

**예시**: **6개**가 나옵니다. 화살표 방향이 3-1 과 **반대**라는 점에 주의하세요.

> `MATCH (c:Course)-[:PREREQ_OF]->(:Course) RETURN c.name` 으로 써도 같은 이름들이 나오지만, 그쪽은 **다음 과목이 둘이면 그 과목이 두 줄로 나옵니다.** `WHERE` 에 적으면 과목당 한 줄입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 3-1 과 같은 자리에 패턴을 적되 NOT 을 빼고 화살표 방향을 뒤집는다.

세부구현:
1. MATCH (c:Course) 로 과목을 모두 잡는다.
2. WHERE (c)-[:PREREQ_OF]->(:Course) 를 적는다. c 에서 나가는 화살표가 하나라도 있으면 남는다.
3. 과목 이름을 별칭 name 으로 반환한다. 결과를 rows3_2 에 담는다.
```

</details>

In [ ]:
rows3_2 = run_cypher("""
MATCH (c:Course)
WHERE (c)-[:PREREQ_OF]->(:Course)
RETURN c.name AS name
""")

In [ ]:
# [자가채점]
assert sorted(r['name'] for r in rows3_2) == ['CS101 프로그래밍입문', 'CS201 자료구조', 'CS202 알고리즘', '수학101 미적분학', '수학201 선형대수', '수학202 확률통계'], \
    'NOT 을 빼고 패턴만 적었는지, 화살표가 (c)-[:PREREQ_OF]->(:Course) 방향인지, 별칭이 name 인지 확인하세요'
assert len(rows3_2) == 6, \
    '과목마다 한 줄이어야 합니다. MATCH 로 이어 붙이면 다음 과목이 둘인 과목이 두 줄로 나옵니다'
print('✅ 통과!')

---
# 4. 정렬과 쪽 넘기기

정렬해 상위를 뽑고 그다음 쪽으로 넘어갑니다(교안_02 5절).

## 4-1. 정렬해 상위만: `ORDER BY`·`LIMIT`
**배경**: 학점이 높은 과목 상위 세 개를 뽑습니다. 학점이 같은 동점이 있으니 순서를 고정할 보조 기준이 필요합니다.

**요구사항**:
- 과목을 **`credits` 내림차순**으로, **동점이면 이름 오름차순**으로 정렬해 **상위 3개**(`LIMIT 3`)를 뽑으세요.
- 결과를 변수 **`rows4_1`** 에 담으세요. 과목 이름을 반환 컬럼 별칭 **`name`** 으로.

**예시**: **3개** 과목이 순서대로 나옵니다(상위권은 학점이 모두 같아 이름순으로 갈립니다).

<details><summary>힌트</summary>

```text
접근방법:
- ORDER BY 에 기준 두 개(학점 내림차순, 이름 오름차순)를 주고 LIMIT 로 자른다.

세부구현:
1. 모든 과목을 잡고 이름을 별칭 name 으로 반환한다.
2. ORDER BY 에 credits 내림차순(DESC)과 보조키로 이름 오름차순을 함께 주고, LIMIT 3 으로 상위 셋만 남긴다.
3. 결과를 rows4_1 에 담는다.
```

</details>

In [ ]:
rows4_1 = run_cypher("""
MATCH (c:Course)
RETURN c.name AS name
ORDER BY c.credits DESC, name ASC
LIMIT 3
""")

In [ ]:
# [자가채점]
assert [r['name'] for r in rows4_1] == ['CS202 알고리즘', 'CS302 운영체제', 'CS401 머신러닝'], \
    'ORDER BY 에 credits 내림차순과 보조 정렬키(이름 오름차순)를 둘 다 줬는지, LIMIT 3 인지 확인하세요'
print('✅ 통과!')

## 4-2. 그다음 세 개: `SKIP` 으로 쪽 넘기기
**배경**: 4-1 에서 상위 세 과목을 봤습니다. 이번엔 **그다음 세 과목**(4~6위)을 봅니다. 목록을 세 개씩 끊어 보여 주는 **2페이지**를 만드는 셈입니다.

**요구사항**:
- 정렬 기준은 **4-1 과 똑같이**: `credits` 내림차순, **동점이면 이름 오름차순**.
- 앞의 **3개를 건너뛰고**(`SKIP 3`) **3개만**(`LIMIT 3`) 뽑으세요.
- 결과를 변수 **`rows4_2`** 에 담으세요. 과목 이름을 반환 컬럼 별칭 **`name`** 으로.

**주의**: 두 페이지의 `ORDER BY` 가 **완전히 같아야** 하고, 보조 정렬키(이름)가 있어야 순서가 하나로 정해집니다. 이 데이터에는 학점이 같은 과목이 여럿이라, 보조 정렬키를 빼면 그 과목들 사이의 순서가 **정해지지 않습니다.** 그러면 1페이지에 나온 과목이 2페이지에 또 나오거나, 어떤 과목은 어느 쪽에도 안 나올 수 있습니다.

**예시**: 3학점 과목 **3개**가 나옵니다.

<details><summary>힌트</summary>

```text
접근방법:
- 4-1 과 똑같이 정렬한 뒤, 앞의 세 개를 건너뛰고 세 개만 남긴다.

세부구현:
1. 모든 과목을 잡고 이름을 별칭 name 으로 반환한다.
2. ORDER BY 를 4-1 과 완전히 똑같이 준다(학점 내림차순, 보조키로 이름 오름차순).
3. SKIP 으로 앞 세 개를 건너뛰고 LIMIT 로 세 개만 남긴다(적는 순서는 ORDER BY, SKIP, LIMIT).
4. 결과를 rows4_2 에 담는다.
```

</details>

In [ ]:
rows4_2 = run_cypher("""
MATCH (c:Course)
RETURN c.name AS name
ORDER BY c.credits DESC, name ASC
SKIP 3
LIMIT 3
""")

In [ ]:
# [자가채점]
assert [r['name'] for r in rows4_2] == ['CS101 프로그래밍입문', 'CS201 자료구조', 'CS301 데이터베이스'], \
    'ORDER BY 를 4-1 과 완전히 똑같이(보조 정렬키 포함) 줬는지, SKIP 3 LIMIT 3 인지 확인하세요'
print('✅ 통과!')

---
수고했어요! LV1 에서 가변길이(`*1..2`·`*0..2`)·`shortestPath`·`length(p)`·`IN`·`CONTAINS`·`STARTS WITH`·`ENDS WITH`·`<>`·정규식 `=~`·`IN $names`·`OR`·`WHERE` 에 적는 패턴 술어(`NOT` 을 붙인 것과 안 붙인 것)·`ORDER BY`·`LIMIT`·`SKIP` 을 **하나씩** 익혔습니다. LV2 에서는 이것들을 **조합**해 물류 배송망을 분석합니다.